### Esse repositório possui função de criação dos datasets de imagens preparados HOG

In [4]:
# Importando libs

from skimage import data
from skimage.color import rgb2gray
from skimage.transform import resize
from skimage.feature import hog
import matplotlib.pyplot as plt
import os
import cv2
import numpy as np
import pandas as pd

In [5]:
# 1. Configurações de Entrada, Saída e Parâmetros
PATHDIR = "data/"
OUTPUT_DIR = "output/"  # Pasta onde os arquivos CSV serão salvos
SEARCH_TERMS = ["Japanese_chin", "Keeshond", "Russian_Blue", "British_Shorthair"]

IMG_SIZES = [256, 128]
CELLS_BLOCKS = [32, 20, 16, 8]


# 2. Pré-processamento Dinâmico
def img_preprocess(img, target_size: int):
    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    img_resized = resize(
        img_rgb, (target_size, target_size), anti_aliasing=True
    )
    return rgb2gray(img_resized)


# 3. Identificação da Classe
def get_class_label(filename: str) -> str:
    fn_lower = filename.lower()
    if any(dog in fn_lower for dog in ["japanese_chin", "keeshond"]):
        return "dog"
    elif any(
        cat in fn_lower
        for cat in ["russian_blue", "british_shorthair", "abyssinian"]
    ):
        return "cat"
    return "unknown"


# 4. Pipeline Principal com Diretório de Saída
def generate_all_hog_datasets(
    path: str, output_path: str, search_list: list, sizes: list, cells: list
):
    # Garante que o diretório de saída existe (cria se não existir)
    os.makedirs(output_path, exist_ok=True)

    # Carrega e filtra as imagens uma única vez na memória
    raw_images = []
    for archive in os.listdir(path):
        archive_lower = archive.lower()
        if any(term.lower() in archive_lower for term in search_list):
            if archive_lower.endswith((".jpg", ".png", ".jpeg")):
                full_path = os.path.join(path, archive)
                img = cv2.imread(full_path)
                if img is not None:
                    raw_images.append((archive, img))

    print(f"Total de {len(raw_images)} imagens carregadas para processamento.")

    # Iteração sobre todas as combinações de Tamanho e Célula
    for size in sizes:
        for cell in cells:
            hog_features = []
            classes = []

            for filename, img in raw_images:
                gray = img_preprocess(img, target_size=size)

                features = hog(
                    gray,
                    orientations=9,
                    pixels_per_cell=(cell, cell),
                    cells_per_block=(2, 2),
                    visualize=False,
                )

                hog_features.append(features)
                classes.append(get_class_label(filename))

            # Criação do DataFrame
            df = pd.DataFrame(hog_features)
            df.columns = [f"hog_feature_{i+1}" for i in range(df.shape[1])]
            df["classe"] = classes

            # Monta o caminho completo no diretório de saída
            filename = f"HOG_{size}_{cell}x{cell}.csv"
            full_output_path = os.path.join(output_path, filename)

            # Salva o CSV na pasta especificada
            df.to_csv(full_output_path, index=False)
            print(f"Dataset gerado com sucesso: {full_output_path}")


# Execução do script
generate_all_hog_datasets(
    PATHDIR, OUTPUT_DIR, SEARCH_TERMS, IMG_SIZES, CELLS_BLOCKS
)

Total de 800 imagens carregadas para processamento.
Dataset gerado com sucesso: output/HOG_256_32x32.csv
Dataset gerado com sucesso: output/HOG_256_20x20.csv


KeyboardInterrupt: 